# 06 Challenger Model Benchmarking

**Author:** Rowan Walker

This notebook presents a benchmarking comparison to a non-trivial challenger model designed to act as a benchmark. The model chosen here is a cross-sectional XGBoost model.

### 6.1 Imports

In [43]:
import numpy as np
import pandas as pd

from src.data.preprocess import generate_valid_tickers, generate_model_inputs
from src.inference.inference import model_inference
from src.training.train import train_model
from src.models.challenger import train_xgb_challenger, build_challenger_inference_data
from src.utils.backtest import backtest, optimise_sharpe
from src.utils.seed import set_global_seed

from src.common.config import CommonConfig, load_yaml, get_config_path
from src.data.config import load_data_config

data_cfg = load_data_config(get_config_path("data.yaml"))

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")
warnings.filterwarnings("ignore", category=UserWarning, module="hmmlearn")
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

%reload_ext autoreload
%autoreload 2

# Global seed setting for reproducibility
set_global_seed()

### 6.2 Run XGBoost Challenger model

#### Setting a common parameter grid for the back test

In [44]:
param_grid = {
    'long_threshold': 0.6, 
    'short_threshold': 0.4, 
    'target_vol': 0.2,  
    'slippage': 1.0,
    'commission': 1.0,
    'take_profit': 0.16,
    'stop_loss': -0.02,
    'max_hold_days': 18.0,
    'max_drawdown': 0.2,
    'leverage': 2.0}

#### Defining end-to-end model runs for model and challenger

In [45]:
def e2e(param_grid):
    for i, t in enumerate((1, 5, 21)):
        valid_tickers = generate_valid_tickers(
            start_date='2010-12-01',
            end_date='2025-12-01')
        
        X, y, stock_ids, regime_X, full_df = generate_model_inputs(
            tickers=valid_tickers,
            train_start='2010-12-01',
            train_end='2019-12-01',
            hold_days=t,
            verbose=False
        )

        train_model(
            X, 
            y, 
            stock_ids, 
            regime_X, 
            hold_days=t,
            verbose=False
        )

    model_inference(
        valid_tickers, 
        full_df, 
        feature_dim=X.shape[2], 
        hold_days=(1,5,21),
        verbose=False
    )
    
    sharpe = backtest(param_grid, start_date='2020-01-01', end_date='2025-12-31', sharpe_only=True)

    return sharpe

In [48]:
def e2e_challenger(param_grid):  
    results = {}
    
    for t in (1, 5, 21):
        file_path = data_cfg['paths']['processed']
        file_name = f'processed_{t}.parquet'
        df = pd.read_parquet(file_path / file_name)
        
        predicted = train_xgb_challenger(
            df=df,
            train_start='2010-01-01',
            train_end='2019-12-31',
            test_start='2020-01-01',
            test_end='2025-12-31',
            verbose=False
        )
    
        results[f'Prediction_{t}'] = predicted
    
    inference_path = data_cfg['paths']['processed']
    inference_file = 'processed_1.parquet' # Any processed dataframe  can be used; label is dropped as a column
    processed_df = pd.read_parquet(inference_path / inference_file)
    
    build_challenger_inference_data(
        probabilities = pd.DataFrame(results),
        processed_df = processed_df, 
        test_start='2020-01-01',
        test_end='2025-12-31'
    )
    
    sharpe = backtest(
        param_grid, 
        start_date='2020-01-01', 
        end_date='2025-12-31',
        sharpe_only=True,
        challenger=True
    )

    return sharpe

### 6.3 Results

In [55]:
results = {}

results['Champion'] = round(e2e(param_grid), 2)
results['Challenger'] = round(e2e_challenger(param_grid), 2)

pd.DataFrame(results, index=['Sharpe'])

,Champion,Challenger
Sharpe,3.45,1.35


**Conclusion**: The Transformer Champion model achieves a Sharpe ratio of 3.45, significantly outperforming the XGBoost Challenger’s Sharpe of 1.35.

This shows that:

- The Transformer captures far more of the underlying signal structure and delivers superior risk-adjusted returns.
- The XGBoost model fails to match the temporal and cross-asset relationships learned by the Transformer.
- The large performance gap suggests the Transformer is both more predictive and more robust, while the XGBoost model likely underfits the complexity of the data.

The Transformer remains the stronger and more production-ready model. The XGBoost challenger does not currently provide a viable alternative and would require major feature or model improvements to become competitive.